# Fuentes oficiales

Audita CSVs curados y trazabilidad de condiciones/componentes.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'Suplematch-Backend').exists():
    ROOT = ROOT / 'Suplematch-Backend'
while ROOT.name != 'Suplematch-Backend' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

KNOWLEDGE = ROOT / 'data/knowledge'
assert KNOWLEDGE.exists()
ROOT

## Archivos

In [ ]:
files = sorted(KNOWLEDGE.glob('*.csv'))
summary = []
for path in files:
    df = pd.read_csv(path, keep_default_na=False)
    summary.append({'file': path.name, 'rows': len(df), 'columns': len(df.columns)})
pd.DataFrame(summary)

## Validación

In [ ]:
sources = pd.read_csv(KNOWLEDGE / 'sources.csv', keep_default_na=False)
conditions = pd.read_csv(KNOWLEDGE / 'conditions.csv', keep_default_na=False)
links = pd.read_csv(KNOWLEDGE / 'condition_component_links.csv', keep_default_na=False)
rules = pd.read_csv(KNOWLEDGE / 'condition_rules.csv', keep_default_na=False)

condition_codes = set(conditions['condition_code'])
source_ids = set(sources['source_id'])
link_conditions = set(links['condition_code'])
rule_conditions = set(rules['condition_code'])

checks = {
    'unknown_link_conditions': sorted(link_conditions - condition_codes),
    'unknown_rule_conditions': sorted(rule_conditions - condition_codes),
    'condition_count': len(condition_codes),
    'source_count': len(source_ids),
    'link_count': len(links),
    'rule_count': len(rules),
}
checks

## Reporte oficial

In [ ]:
import sys

sys.path.insert(0, str(ROOT))
from scripts.training.entrenar_modelo_condiciones import load_knowledge, validate_knowledge

validate_knowledge(load_knowledge())